# 🎬 AutoScene Studio — GPU Renderer (Colab)
Runs your **existing AutoScene web app** on a **Colab GPU** (NVENC hardware
encoding = ~10x faster), with **Edge / Fish / Kokoro** voices, saves finished
videos to **Google Drive**, and exposes the dashboard via a **Cloudflare
tunnel**. No login required.

## How to use
1. Run each cell in order (▶ Run all). A popup may ask to grant **Drive access**
   — click Allow (that's where videos get saved).
2. After the **Cloudflare** cell you'll see a `https://xxx.trycloudflare.com` URL
   — **open it**. That's your AutoScene dashboard.
3. Upload media + script, pick a voice (Edge/Fish/Kokoro), hit render. Finished
   video saves to `MyDrive/AutoScene_Outputs/`.
4. The **last cell plays silent audio** so Colab doesn't disconnect mid-render
   (it keeps the tab "active"). Keep that tab open.


---
## Step 1 — Setup: Drive + repos + engine venv + GPU ffmpeg


In [ ]:

# Mount Google Drive (finished videos are saved here)
from google.colab import drive
drive.mount('/content/drive')
import os
print("Drive mounted:", os.path.exists('/content/drive/MyDrive'))


In [ ]:

import os, subprocess
print("AutoScene GPU notebook — v5")
# --- clone the working engine + web app ---
os.chdir('/content')
def clone(url, dest):
    if not os.path.exists(dest):
        subprocess.run(['git','clone','-q','--depth','1',url,dest], check=True)
        print("cloned", dest)
clone('https://github.com/5656wwed/automan-engine', '/content/automan-engine')
clone('https://github.com/5656wwed/automan-web-colab', '/content/automan-web')
print("repos ready")


In [ ]:

# --- engine deps: install into base Colab Python (a venv is not needed here) ---
import os, subprocess, sys
os.chdir('/content')
PYV = sys.executable                       # Colab's base python (has torch)
open('/content/PYV.txt', 'w').write(PYV)   # used by the launch cell later
deps = ['edge-tts', 'openai', 'aiohttp', 'pydub', 'Pillow',
        'python-dotenv', 'typer', 'rich', 'pydantic', 'requests',
        'fastapi', 'uvicorn']
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + deps,
                   capture_output=True, text=True)
if r.returncode != 0:
    print('⚠ some deps failed to install (see below) — continuing anyway')
    print(r.stderr[-500:])
print('interpreter:', PYV)
print('deps installed')


In [ ]:

# --- ffmpeg: use the SYSTEM (apt) build, like the dubbing — it loads libcuda
#     for NVENC, whereas the static BtbN build cannot (Cannot load libcuda.so.1).
#     GPU is auto-detected: if a GPU is present, use h264_nvenc; else libx264. ---
import os, shutil, subprocess
os.chdir('/content')
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False  # CPU-only runtime: nvidia-smi not installed
print('GPU (nvidia-smi):', has_gpu)
ff = shutil.which('ffmpeg') or '/usr/bin/ffmpeg'
fp = shutil.which('ffprobe') or '/usr/bin/ffprobe'
if not os.path.exists(ff):
    subprocess.run('apt-get install -y -qq ffmpeg', shell=True, check=False)
    ff = shutil.which('ffmpeg') or '/usr/bin/ffmpeg'
print('ffmpeg:', subprocess.run([ff, '-version'], capture_output=True, text=True).stdout.splitlines()[0])
enc = subprocess.run([ff, '-hide_banner', '-encoders'], capture_output=True, text=True).stdout
nvenc = enc.count('nvenc')
# NVENC self-test (only meaningful if a GPU is present)
_t = subprocess.run([ff, '-v', 'error', '-f', 'lavfi', '-i',
                     'testsrc=size=320x180:rate=15', '-frames:v', '1',
                     '-c:v', 'h264_nvenc', '-y', '/tmp/_nv_test.mp4'],
                    capture_output=True, text=True)
works = has_gpu and os.path.exists('/tmp/_nv_test.mp4')
open('/content/nvenc_ok', 'w').write('1' if (works and nvenc) else '0')
open('/content/FFMPEG_PATH', 'w').write(ff)
open('/content/FFPROBE_PATH', 'w').write(fp)
print('nvenc encoders:', nvenc, '| NVENC self-test:', 'WORKS ✅' if works else 'FAILS ❌')
if has_gpu and not works:
    print((_t.stderr or '').strip()[-400:])
print('encoder:', 'h264_nvenc (GPU)' if works and nvenc else 'libx264 (CPU)')
print('using:', ff)


In [ ]:

# --- patch the engine: NVENC encode + Kokoro provider ---
import subprocess, os
os.chdir('/content')
PATCH = "#!/usr/bin/env python3\n\"\"\"Patch a fresh clone of theautoman for the Colab GPU build.\n\nRun from the repo root (or pass repo root as argv[1]):\n    python3 patch_engine.py [/path/to/theautoman]\n\nDoes three things:\n 1. NVENC: adds a `_video_flags(export)` helper to app/ffmpeg/wrapper.py and\n    app/transitions/engine.py so h264_nvenc gets correct flags when\n    AUTOMAN_CODEC=h264_nvenc. Makes config codec read $AUTOMAN_CODEC.\n 2. Kokoro: writes app/tts/kokoro_provider.py (OpenAI-compatible local server).\n 3. Registers the Kokoro provider in voice_manager.py and adds the enum.\n\"\"\"\nimport re\nimport shutil\nimport sys\nfrom pathlib import Path\n\nROOT = Path(sys.argv[1]) if len(sys.argv) > 1 else Path.cwd()\n\nHELPER = '''\ndef _video_flags(export):\n    \"\"\"Return codec flags for export. Uses NVENC flags when codec is an nvenc\n    codec (h264_nvenc / hevc_nvenc), otherwise the original x264 flags.\"\"\"\n    codec = (export.codec or \"libx264\").lower()\n    if \"nvenc\" in codec:\n        q = getattr(export, \"quality\", None)\n        qv = getattr(q, \"value\", None) if q is not None else None\n        nv_preset = {\"low\": \"p1\", \"medium\": \"p3\", \"high\": \"p4\", \"ultra\": \"p6\"}.get(qv, \"p4\")\n        return [\"-c:v\", export.codec, \"-rc\", \"vbr\", \"-cq\", str(export.crf),\n                \"-b:v\", \"0\", \"-preset\", nv_preset]\n    return [\"-c:v\", export.codec, \"-crf\", str(export.crf), \"-preset\", export.preset_speed]\n'''\n\nKOKORO_PROVIDER = '''\"\"\"Kokoro TTS provider (OpenAI-compatible local/GPU server).\"\"\"\nfrom __future__ import annotations\n\nimport asyncio\nimport os\nfrom pathlib import Path\nfrom typing import Optional\n\nfrom app.tts.base import TTSProvider, TTSResult, Voice\nfrom app.utils.logger import get_logger\n\nlog = get_logger(\"tts.kokoro\")\n\nKOKORO_VOICES = [\n    (\"af_heart\", \"Heart (US female)\", \"female\"),\n    (\"af_bella\", \"Bella (US female)\", \"female\"),\n    (\"af_nicole\", \"Nicole (US female)\", \"female\"),\n    (\"af_sarah\", \"Sarah (US female)\", \"female\"),\n    (\"af_sky\", \"Sky (US female)\", \"female\"),\n    (\"am_adam\", \"Adam (US male)\", \"male\"),\n    (\"am_michael\", \"Michael (US male)\", \"male\"),\n    (\"am_fenrir\", \"Fenrir (US male)\", \"male\"),\n    (\"bf_emma\", \"Emma (UK female)\", \"female\"),\n    (\"bf_isabella\", \"Isabella (UK female)\", \"female\"),\n    (\"bm_george\", \"George (UK male)\", \"male\"),\n    (\"bm_lewis\", \"Lewis (UK male)\", \"male\"),\n    (\"ff_siwis\", \"Siwis (FR female)\", \"female\"),\n]\n\n\nclass KokoroProvider(TTSProvider):\n    provider_name = \"kokoro\"\n\n    def __init__(self, base_url: Optional[str] = None):\n        self._base_url = base_url or os.environ.get(\"KOKORO_URL\", \"http://localhost:8002/v1\")\n        self._model = os.environ.get(\"KOKORO_MODEL\", \"kokoro\")\n        self._client = None\n\n    def _get_client(self):\n        if self._client is None:\n            from openai import OpenAI\n            self._client = OpenAI(base_url=self._base_url,\n                                  api_key=os.environ.get(\"KOKORO_API_KEY\", \"kokoro\"))\n        return self._client\n\n    def is_available(self) -> bool:\n        return True\n\n    async def generate(self, text, output_path, voice_id=\"af_heart\", speed=1.0,\n                       pitch=1.0, stability=0.5, emotion=None, **kwargs):\n        output_path = Path(output_path)\n        output_path.parent.mkdir(parents=True, exist_ok=True)\n\n        def _gen():\n            self._get_client().audio.speech.create(\n                model=self._model, voice=voice_id, input=text,\n                speed=speed, response_format=\"mp3\").stream_to_file(str(output_path))\n\n        loop = asyncio.get_event_loop()\n        await loop.run_in_executor(None, _gen)\n        from app.utils.audio import get_audio_duration\n        d = get_audio_duration(output_path)\n        log.info(f\"Generated Kokoro audio: {output_path.name} ({d:.1f}s)\")\n        return TTSResult(audio_path=output_path, duration=d, voice_id=voice_id, text=text)\n\n    async def list_voices(self) -> list[Voice]:\n        return [Voice(voice_id=v, name=n, provider=self.provider_name, language=\"en\", gender=g)\n                for v, n, g in KOKORO_VOICES]\n'''\n\n\ndef patch_wrapper(path: Path) -> bool:\n    src = path.read_text()\n    changed = False\n    pat = re.compile(r'\"-c:v\",\\s*export\\.codec,\\s*\"-crf\",\\s*str\\(export\\.crf\\),\\s*\"-preset\",\\s*export\\.preset_speed,', re.S)\n    if \"_video_flags(export)\" not in src:\n        src, n = pat.subn(\"*_video_flags(export),\", src)\n        if n:\n            changed = True\n    if \"def _video_flags\" not in src:\n        src = src.rstrip() + \"\\n\" + HELPER + \"\\n\"\n        changed = True\n    if changed:\n        path.write_text(src)\n        print(f\"[wrapper] _video_flags ensured\")\n    return changed\n\n\ndef patch_engine_transitions(path: Path) -> bool:\n    if not path.exists():\n        return False\n    src = path.read_text()\n    changed = False\n    pat = re.compile(r'\"-c:v\",\\s*export\\.codec,\\s*\"-crf\",\\s*str\\(export\\.crf\\),\\s*\"-preset\",\\s*export\\.preset_speed,', re.S)\n    if \"_video_flags(export)\" not in src:\n        src, n = pat.subn(\"*_video_flags(export),\", src)\n        if n:\n            changed = True\n    if \"def _video_flags\" not in src:\n        src = src.rstrip() + \"\\n\" + HELPER + \"\\n\"\n        changed = True\n    if changed:\n        path.write_text(src)\n        print(f\"[transitions] _video_flags ensured\")\n    return changed\n\n\ndef patch_config(path: Path) -> bool:\n    src = path.read_text()\n    changed = False\n    if \"AUTOMAN_CODEC\" not in src:\n        src = src.replace('codec: str = \"libx264\"',\n                          'codec: str = os.environ.get(\"AUTOMAN_CODEC\", \"libx264\")', 1)\n        if \"import os\" not in src.split(\"class ExportSettings\")[0]:\n            src = src.replace(\"from __future__ import annotations\",\n                              \"from __future__ import annotations\\nimport os\", 1)\n        changed = True\n    if 'POCKET = \"pocket\"' in src and 'KOKORO = \"kokoro\"' not in src:\n        src = src.replace('    POCKET = \"pocket\"', '    POCKET = \"pocket\"\\n    KOKORO = \"kokoro\"', 1)\n        changed = True\n    if changed:\n        path.write_text(src)\n        print(\"[config] codec->env + KOKORO enum added\")\n    return changed\n\n\ndef patch_voice_manager(path: Path) -> bool:\n    src = path.read_text()\n    if \"kokoro_provider\" in src:\n        return False\n    src = src.replace(\n        \"from app.tts.fish_audio_provider import FishAudioProvider\",\n        \"from app.tts.fish_audio_provider import FishAudioProvider\\nfrom app.tts.kokoro_provider import KokoroProvider\", 1)\n    src = src.replace(\n        'self._providers[\"inworld\"] = inworld',\n        'self._providers[\"inworld\"] = inworld\\n        self._providers[\"kokoro\"] = KokoroProvider()', 1)\n    path.write_text(src)\n    print(\"[voice_manager] Kokoro registered\")\n    return True\n\n\ndef main():\n    (ROOT / \"app\" / \"tts\" / \"kokoro_provider.py\").write_text(KOKORO_PROVIDER)\n    print(\"[kokoro_provider.py] written\")\n    patch_wrapper(ROOT / \"app\" / \"ffmpeg\" / \"wrapper.py\")\n    patch_engine_transitions(ROOT / \"app\" / \"transitions\" / \"engine.py\")\n    patch_config(ROOT / \"app\" / \"core\" / \"config.py\")\n    patch_voice_manager(ROOT / \"app\" / \"tts\" / \"voice_manager.py\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
open('/content/patch_engine.py','w').write(PATCH)
r = subprocess.run(['python3','/content/patch_engine.py','/content/automan-engine'], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-400:] if r.returncode else "OK")


---
## Step 2 — Start Kokoro TTS (GPU) + the web app + Cloudflare tunnel


In [ ]:

# --- Kokoro TTS server (runs in base Colab Python which has torch+CUDA) ---
import os, subprocess, sys, time, requests, threading
KOKORO_SRC = "\"\"\"Minimal Kokoro TTS server (OpenAI-compatible /v1/audio/speech) for Colab GPU.\n\nRuns in the base Colab Python (which has torch+CUDA preinstalled). Exposes\nPOST /v1/audio/speech and GET /v1/models so the engine's KokoroProvider\n(an OpenAI-compatible client) can call it.\n\nUsage:\n    python kokoro_server.py [--port 8002]\n\"\"\"\nimport argparse\nimport io\nimport os\nimport subprocess\nimport tempfile\n\nimport torch\n\n# Load the model once at startup (GPU if available).\n_DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n_pipelines = {}  # lang_code -> KPipeline\n\n\ndef _get_pipeline(lang: str):\n    if lang not in _pipelines:\n        from kokoro import KPipeline\n        _pipelines[lang] = KPipeline(lang_code=lang, repo_id=\"hexgrad/Kokoro-82M\",\n                                     device=_DEVICE)\n    return _pipelines[lang]\n\n\ndef _lang_for_voice(voice: str) -> str:\n    v = (voice or \"af_heart\").lower()\n    if v.startswith(\"bf_\") or v.startswith(\"bm_\"):\n        return \"b\"\n    if v.startswith(\"ff_\") or v.startswith(\"fm_\"):\n        return \"f\"\n    return \"a\"\n\n\ndef _wav_bytes(text, voice, speed) -> bytes:\n    pipe = _get_pipeline(_lang_for_voice(voice))\n    chunks = []\n    sr = 24000\n    for result in pipe(text, voice=voice, speed=float(speed)):\n        chunks.append(result.audio)\n    if not chunks:\n        raise RuntimeError(\"Kokoro produced no audio\")\n    audio = torch.cat(chunks).detach().cpu().numpy()\n    fd, wav = tempfile.mkstemp(suffix=\".wav\")\n    try:\n        import soundfile as sf\n        sf.write(wav, audio, sr)\n        mp3 = wav[:-4] + \".mp3\"\n        ff = os.environ.get(\"KOKORO_FFMPEG\", \"/content/ffmpeg/ffmpeg\")\n        subprocess.run([ff, \"-y\", \"-loglevel\", \"error\", \"-i\", wav,\n                        \"-b:a\", \"192k\", mp3], check=True)\n        return open(mp3, \"rb\").read()\n    finally:\n        try:\n            os.close(fd)\n            os.remove(wav)\n            if os.path.exists(wav[:-4] + \".mp3\"):\n                os.remove(wav[:-4] + \".mp3\")\n        except Exception:\n            pass\n\n\nfrom fastapi import FastAPI, Request, Response\nfrom fastapi.responses import JSONResponse\nimport uvicorn\n\napp = FastAPI(title=\"Kokoro TTS (Colab)\")\n\n\n@app.get(\"/v1/models\")\nasync def models():\n    return {\"object\": \"list\", \"data\": [{\"id\": \"kokoro\", \"object\": \"model\"}]}\n\n\n@app.post(\"/v1/audio/speech\")\nasync def speech(req: Request):\n    body = await req.json()\n    text = body.get(\"input\", \"\")\n    voice = body.get(\"voice\", \"af_heart\")\n    speed = body.get(\"speed\", 1.0)\n    if not text:\n        return JSONResponse({\"error\": \"empty input\"}, status_code=400)\n    audio = _wav_bytes(text, voice, speed)\n    return Response(content=audio, media_type=\"audio/mpeg\")\n\n\nif __name__ == \"__main__\":\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--port\", type=int, default=int(os.environ.get(\"KOKORO_PORT\", \"8002\")))\n    args = ap.parse_args()\n    print(f\"Kokoro server on {_DEVICE} :{args.port}\", flush=True)\n    uvicorn.run(app, host=\"0.0.0.0\", port=args.port, log_level=\"info\")\n"
open('/content/kokoro_server.py','w').write(KOKORO_SRC)
# install kokoro into the base python (torch already present)
subprocess.run([sys.executable,'-m','pip','install','-q','kokoro','soundfile','fastapi','uvicorn'], check=True)
env = dict(os.environ); env['KOKORO_FFMPEG']='/content/ffmpeg/ffmpeg'; env['KOKORO_PORT']='8002'
open('/content/kokoro.log','w').close()
p = subprocess.Popen([sys.executable,'/content/kokoro_server.py'], env=env,
                     stdout=open('/content/kokoro.log','a'), stderr=subprocess.STDOUT, cwd='/content')
# wait for it to come up
for _ in range(120):
    try:
        r = requests.get('http://localhost:8002/v1/models', timeout=2)
        if r.status_code == 200: break
    except Exception: pass
    time.sleep(3)
print("Kokoro server up:", os.path.exists('/content/kokoro.log'))
print(open('/content/kokoro.log').read()[-300:])


In [ ]:

# --- launch the AutoScene web app on a local port ---
import os, subprocess, time, requests
WEB = '/content/automan-web'
PYV = open('/content/PYV.txt').read().strip()  # engine interpreter (venv or base)
os.makedirs('/content/drive/MyDrive/AutoScene_Outputs', exist_ok=True)
ffmpeg_bin = open('/content/FFMPEG_PATH').read().strip()
ffprobe_bin = open('/content/FFPROBE_PATH').read().strip()
codec = 'h264_nvenc' if open('/content/nvenc_ok').read().strip() == '1' else 'libx264'
env = dict(os.environ)
env.update({
  'THEAUTOMAN_DIR': '/content/automan-engine',
  'THEAUTOMAN_PYTHON': PYV,
  'AUTOMAN_CODEC': codec,
  'FFMPEG_PATH': ffmpeg_bin,
  'FFPROBE_PATH': ffprobe_bin,
  'DRIVE_OUT_DIR': '/content/drive/MyDrive/AutoScene_Outputs',
  'KOKORO_URL': 'http://localhost:8002/v1',
  'POCKET_VOICES_DIR': '/content/pocket_voices',
  'LOG_PATH': '/content/webapp.log',
})
# NVENC self-test passed in the ffmpeg cell -> codec is h264_nvenc; else libx264.
print('ffmpeg:', ffmpeg_bin)
print('encoder:', codec)
open('/content/webapp.log','w').close()
p = subprocess.Popen([PYV,'-m','uvicorn','app.main:app','--host','0.0.0.0','--port','7860'],
                     cwd=WEB, env=env, stdout=open('/content/webapp.log','a'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        r = requests.get('http://localhost:7860/health', timeout=2)
        if r.status_code == 200: break
    except Exception: pass
    time.sleep(2)
print("web app health:", requests.get('http://localhost:7860/health', timeout=5).status_code)
print(open('/content/webapp.log').read()[-300:])


In [ ]:

# --- Cloudflare tunnel -> public dashboard URL ---
import subprocess, time, re
# install cloudflared if missing
subprocess.run(['which','cloudflared'], capture_output=True)
import shutil
if not shutil.which('cloudflared'):
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod','+x','/usr/local/bin/cloudflared'])
subprocess.run(['pkill','-f','cloudflared'], capture_output=True)
time.sleep(1)
open('/content/tunnel.log','w').close()
p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:7860','--no-autoupdate'],
                     stdout=open('/content/tunnel.log','a'), stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    txt = open('/content/tunnel.log', errors='ignore').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m: url = m.group(0); break
    time.sleep(3)
print("\n\n🖥  DASHBOARD URL:", url or "(still connecting... check /content/tunnel.log)")
print("Open it in your browser — no login needed.")


---
## Step 3 — Keep the session alive (silence audio so Colab doesn't disconnect)


In [ ]:
%%javascript
// Play silent audio on a loop so Colab keeps the tab 'active' and
// doesn't disconnect the session during long renders.
(function(){
  var b64 = '...';
  var a = new Audio('data:audio/wav;base64,'+b64);
  a.loop = true; a.volume = 0.01;
  a.play();
  setInterval(function(){ if(a.paused){ a.play(); } }, 30000);
  console.log('Keep-alive silence audio playing');
})();
